# Phase 5｜训练数据规模实验与最终模型选择

**核心：比较不同训练数据规模，在同一份验证集上选出最终模型，再进行正式测试。**

| 步骤 | 要做什么 | 用到的技术与知识 | 主要产出 |
|---|---|---|---|
| **5.1 实验准备与数据核对** | 核对 Phase 3 真实训练配置；准备三档训练数据；固定 Val 1,000，继续封存 Test 2,000 | 控制变量、分层抽样、数据去重、防泄漏检查 | 实验配置表、三档训练数据 |
| **5.2 训练规模实验** | 保留 E1，分别训练 E2、E3；统一 Rank=16 和其余训练规则 | SFT、标准 LoRA、回答部分 Loss、梯度累积 | 各实验 adapter、训练日志 |
| **5.3 验证与收敛分析** | 在同一份 Val 上比较准确率、Loss、字段提取效果、分类表现和训练耗时 | 独立生成评测、Accuracy/F1/Recall、Loss 曲线、过拟合判断 | 实验对比表、曲线、错误分析 |
| **5.4 模型选择与最终测试** | 根据 Val 表现和训练成本选定模型，冻结配置，再评测 Test 2,000；回归检查困难样本 | 验证集选模型、独立测试、边界能力分析 | 最终模型配置、Test 结果与局限说明 |

研究训练数据从 1,600 条增加到 4,000 条、8,000 条后，Task A 和 Task B 的表现如何变化，并根据同一份 Validation 选择最终模型。

| 实验 | 训练指令数 | LoRA Rank | 状态 |
|---|---:|---:|---|
| E1 | 1,600 | 16 | 已训练，作为参照 |
| E2 | 4,000 | 16 | 待训练 |
| E3 | 8,000 | 16 | 待训练 |

三组实验固定 Val 1,000 条（Task A/B 各 500 条）。Test 2,000 条保持封存，直到模型选定。

我们已经完成了 Phase 4 的基线对比和错误分析，这一阶段重点研究：**训练数据从 1,600 增加到 4,000、8,000 条，效果是否改善，增加的训练成本是否值得。**

## 1. 读取 E1 训练配置

上传 Phase 3 结果包，读取其中的 adapter 配置与实验记录，
核对后续 E2、E3 需要保持一致的训练参数。

In [ ]:
from google.colab import files
from zipfile import ZipFile
import json

# 从电脑上传 Phase 3 结果压缩包
uploaded = files.upload()
zip_name = next(
    (name for name in uploaded if name.endswith(".zip")),
    None
)
if zip_name is None:
    raise ValueError("请上传 FleetMind_LLM_Phase3_Results.zip")

# 直接读取压缩包里的 JSON，不解压大型模型文件
with ZipFile(zip_name) as z:
    required = (
        "best_adapter/adapter_config.json",
        "experiment_manifest.json",
    )
    missing = [name for name in required if name not in z.namelist()]
    if missing:
        raise FileNotFoundError(f"压缩包缺少文件：{missing}")

    adapter = json.loads(z.read(required[0]))
    manifest = json.loads(z.read(required[1]))

print("===== E1 实际配置 =====")
print("基座模型：", adapter["base_model_name_or_path"])
print("LoRA Rank：", adapter["r"])
print("LoRA Alpha：", adapter["lora_alpha"])
print("LoRA Dropout：", adapter["lora_dropout"])
print("目标模块：", adapter["target_modules"])
print("学习率：", manifest["learning_rate"])
print("Batch size：", manifest["batch_size"])
print("梯度累积步数：", manifest["gradient_accumulation_steps"])
print("Epoch：", manifest["epochs"])
print("最大长度：", manifest["max_length"])
print("随机种子：", manifest["seed"])

assert adapter["r"] == 16, "E1 Rank 与预期不符，请检查上传的文件"
assert manifest["lora_rank"] == adapter["r"], "实验记录与 adapter 配置不一致"
print("\nE1 Rank 核对通过。")

### 1.2 检查扩展数据包

列出数据包中的文件及大小，定位 Train、Val、Test。
下一步再核对训练样本数量、A/B 比例及 E1/E2/E3 的包含关系。

In [ ]:
from google.colab import files
from zipfile import ZipFile

uploaded_data = files.upload()

data_zip_name = next(
    (name for name in uploaded_data if name.lower().endswith(".zip")),
    None
)
if data_zip_name is None:
    raise ValueError("请上传 Phase 4 扩展数据集 ZIP")

print("数据包：", data_zip_name)
print("===== 压缩包文件清单 =====")

with ZipFile(data_zip_name) as z:
    for item in z.infolist():
        if not item.is_dir():
            print(f"{item.filename}  |  {item.file_size:,} bytes")

###1.3 定位原始业务数据

Phase 4 数据包只有 1,600 条训练指令。
本节检查原始数据包的文件结构，为生成 E2（4,000 条）
和 E3（8,000 条）寻找 Train 业务记录。

In [ ]:
from google.colab import files
from zipfile import ZipFile

uploaded_source = files.upload()

source_zip_name = next(
    (name for name in uploaded_source if name.lower().endswith(".zip")),
    None
)
if source_zip_name is None:
    raise ValueError("请上传 Fleetmind_dataset.zip")

print("原始数据包：", source_zip_name)
print("===== 文件清单 =====")

with ZipFile(source_zip_name) as z:
    for item in z.infolist():
        if not item.is_dir():
            print(f"{item.filename}  |  {item.file_size:,} bytes")

### 1.4 核对数据来源与训练划分

通过 SHA-256 确认原始业务数据包和 E1 训练文件的身份。
统计业务记录的 Train/Validation/Test 划分，并检查 E1 样本
是否全部来自 Train。这里只检查划分，不读取 Test 答案或评分。

In [ ]:
from zipfile import ZipFile
from collections import Counter
from pathlib import Path
import csv
import hashlib
import io
import json

source_path = Path("Fleetmind_dataset.zip")
phase4_path = Path("FleetMind_LLM_Phase4_Evaluation_Dataset.zip")
assert source_path.is_file() and phase4_path.is_file(), "请确认两个 ZIP 均已上传到当前 Colab 会话"

# 核对原始包是否正是 Phase 4 使用的来源
source_sha = hashlib.sha256(source_path.read_bytes()).hexdigest()
expected_source_sha = "dd171dc31d5d5fed0e010f45a81ded8e404eecff14a78a78702fa8f377457fba"
assert source_sha == expected_source_sha, "原始数据包哈希与 Phase 4 记录不一致"

with ZipFile(phase4_path) as z:
    folder = "fleetmind_phase4_evaluation/"

    # 读取划分表：只使用记录编号及其所属划分
    with z.open(folder + "split_assignments.csv") as f:
        reader = csv.DictReader(io.TextIOWrapper(f, encoding="utf-8-sig"))
        split_by_load = {row["load_id"]: row["split"] for row in reader}

    # 核对 E1 的训练文件与 Phase 3 训练记录一致
    train_bytes = z.read(folder + "train.jsonl")
    train_sha = hashlib.sha256(train_bytes).hexdigest()
    expected_train_sha = "8a0902e922d37262ee95ef747aa9dfad21150d0dca640e0903609a75f833d95c"
    assert train_sha == expected_train_sha, "E1 训练文件哈希与 Phase 3 记录不一致"

    e1_samples = [json.loads(line) for line in train_bytes.splitlines()]

split_counts = Counter(split_by_load.values())
task_counts = Counter(row["task"] for row in e1_samples)
e1_load_ids = {row["source_load_id"] for row in e1_samples}
wrong_split = [
    row["sample_id"] for row in e1_samples
    if split_by_load.get(row["source_load_id"]) != "train"
]

print("原始数据包哈希：通过")
print("E1 训练文件哈希：通过")
print("业务记录划分：", dict(split_counts))
print("E1 指令数：", len(e1_samples))
print("E1 任务分布：", dict(task_counts))
print("E1 使用的不同业务记录数：", len(e1_load_ids))
print("E1 不属于 Train 的样本数：", len(wrong_split))

assert len(e1_samples) == 1600
assert task_counts == {"A": 800, "B": 800}
assert not wrong_split, f"发现非 Train 样本：{wrong_split[:5]}"
print("E1 来源与划分核对通过。")

### 1.5 检查可扩展的 Train 业务记录

读取原始四张表，检查记录数量、关键编号是否重复，
以及 Train 划分中尚未被 E1 使用的业务记录数量。
本节只检查数据，不生成新指令样本。

In [ ]:
from zipfile import ZipFile
import pandas as pd

with ZipFile("Fleetmind_dataset.zip") as z:
    loads = pd.read_csv(z.open("Fleetmind_dataset/loads.csv"))
    trips = pd.read_csv(z.open("Fleetmind_dataset/trips.csv"))
    routes = pd.read_csv(z.open("Fleetmind_dataset/routes.csv"))
    events = pd.read_csv(z.open("Fleetmind_dataset/delivery_events.csv"))

print("===== 原始表 =====")
for name, df in [
    ("loads", loads), ("trips", trips),
    ("routes", routes), ("delivery_events", events)
]:
    print(f"{name}: {len(df):,} 行")

print("\n===== 关键编号检查 =====")
print("loads.load_id 重复数：", loads["load_id"].duplicated().sum())
print("trips.trip_id 重复数：", trips["trip_id"].duplicated().sum())
print("routes.route_id 重复数：", routes["route_id"].duplicated().sum())
print("事件类型：", events["event_type"].value_counts().to_dict())

train_ids = {load_id for load_id, split in split_by_load.items() if split == "train"}
available_ids = train_ids - e1_load_ids

print("\n===== 可扩展范围 =====")
print("Train 业务记录：", len(train_ids))
print("E1 已使用：", len(e1_load_ids))
print("其余 Train 业务记录：", len(available_ids))
print("E2 还需新增业务记录：", 2000 - len(e1_load_ids))
print("E3 相比 E2 还需新增业务记录：", 4000 - 2000)

### 1.6 关联 Train 业务记录

按 load_id、trip_id、route_id 关联原始表，只保留 Delivery 事件。
检查关联后的记录是否一单一行，以及生成 A/B 指令所需字段是否缺失。

In [ ]:
# 每趟业务只取交付事件；Pickup 事件不用于交付延误标签
delivery = events.loc[
    events["event_type"] == "Delivery",
    ["load_id", "trip_id", "scheduled_datetime", "actual_datetime"]
].copy()

# validate 会在关联键不符合预期时直接报错，避免悄悄产生重复行
train_records = (
    loads.loc[loads["load_id"].isin(train_ids)]
    .merge(trips, on="load_id", validate="one_to_one")
    .merge(routes, on="route_id", validate="many_to_one")
    .merge(delivery, on=["load_id", "trip_id"], validate="one_to_one")
)

required = [
    "load_id", "trip_id", "truck_id",
    "origin_city", "destination_city",
    "revenue", "fuel_gallons_used",
    "scheduled_datetime", "actual_datetime",
]

print("关联后的 Train 记录数：", len(train_records))
print("不同 load_id 数：", train_records["load_id"].nunique())
print("关键字段缺失数：")
print(train_records[required].isna().sum().to_string())

unused_records = train_records.loc[
    ~train_records["load_id"].isin(e1_load_ids)
].copy()
print("\n未被 E1 使用的关联记录数：", len(unused_records))

assert train_records["load_id"].is_unique
assert set(e1_load_ids).issubset(set(train_records["load_id"]))

### 1.7 固定三档实验的业务记录

保留 E1 原有的 800 条业务记录；从其余 Train 记录中固定随机种子，
先选 1,200 条加入 E2，再选 2,000 条加入 E3。
检查 E1 ⊂ E2 ⊂ E3，且三组均不包含 Validation 或 Test 记录。

In [ ]:
import random

# 排序后再抽取，使相同数据和种子能得到相同结果
candidate_ids = sorted(set(unused_records["load_id"]))
rng = random.Random(5042)
rng.shuffle(candidate_ids)

new_e2_ids = set(candidate_ids[:1200])
new_e3_ids = set(candidate_ids[1200:3200])

e1_ids = set(e1_load_ids)
e2_ids = e1_ids | new_e2_ids
e3_ids = e2_ids | new_e3_ids

for name, ids, expected in [
    ("E1", e1_ids, 800),
    ("E2", e2_ids, 2000),
    ("E3", e3_ids, 4000),
]:
    outside_train = [load_id for load_id in ids
                     if split_by_load.get(load_id) != "train"]
    print(
        f"{name}: {len(ids)} 条业务记录，"
        f"预计 {len(ids) * 2} 条 A/B 指令，"
        f"非 Train 记录 {len(outside_train)} 条"
    )
    assert len(ids) == expected
    assert not outside_train

assert e1_ids < e2_ids < e3_ids
print("包含关系及划分检查通过：E1 ⊂ E2 ⊂ E3")

###1.8 生成 E2、E3 指令

保留 E1 原始指令，分别为新增的 Train 业务记录生成 Task A 和 Task B。
沿用 E1 的提示词、JSON 字段及训练模板；Task A 的缺失字段填 null。
生成后检查数量与 A/B 比例，再决定是否保存。

In [ ]:
from datetime import datetime
import json
import random
import pandas as pd

# 读取 Phase 1 保存的任务说明和训练模板
with ZipFile(phase4_path) as z:
    info = json.loads(z.read(
        "fleetmind_phase4_evaluation/phase1_dataset_info.json"
    ))

fields = info["task_A_fields"]
labels = {
    "load_id": "运单编号",
    "truck_id": "车辆编号",
    "origin_city": "出发城市",
    "destination_city": "目的城市",
    "revenue_usd": "运单收入（美元）",
    "fuel_used_gallons": "实际耗油量（加仑）",
    "scheduled_delivery": "计划交付时间",
    "actual_delivery": "实际交付时间",
}

# 按 load_id 索引已关联的 Train 记录
records_by_id = train_records.set_index("load_id", drop=False)

def make_pair(load_id, position, omitted_fields):
    row = records_by_id.loc[load_id]
    rng = random.Random(f"phase5-{load_id}")

    values = {
        "load_id": load_id,
        "truck_id": None if pd.isna(row["truck_id"]) else str(row["truck_id"]),
        "origin_city": str(row["origin_city"]),
        "destination_city": str(row["destination_city"]),
        "revenue_usd": float(row["revenue"]),
        "fuel_used_gallons": float(row["fuel_gallons_used"]),
        "scheduled_delivery": str(row["scheduled_datetime"]),
        "actual_delivery": str(row["actual_datetime"]),
    }

    # Task A：只展示未被主动省略、且原始值确实存在的字段
    shown = [key for key in fields
             if key not in omitted_fields and values[key] is not None]
    rng.shuffle(shown)
    details = "；".join(
        f"{labels[key]}：{values[key]}" for key in shown
    ) + "。"

    a_template = info["task_A_templates"]["train"][position % 4]
    a_answer = {
        key: values[key] if key in shown else None
        for key in fields
    }

    common = {
        "source_load_id": load_id,
        "source_trip_id": str(row["trip_id"]),
        "split": "train",
    }
    system = {"role": "system", "content": info["system_prompt"]}

    a = {
        "sample_id": f"{load_id}_A",
        **common,
        "task": "A",
        "messages": [
            system,
            {"role": "user", "content":
                info["task_A_instruction"] + a_template.format(details=details)},
            {"role": "assistant", "content": json.dumps(a_answer, ensure_ascii=False)},
        ],
        "template_id": f"train_A_{position % 4 + 1}",
        "omitted_fields": omitted_fields,
        "shown_fields": shown,
    }

    # Task B：直接比较完整时间；不截断秒或微秒
    scheduled = values["scheduled_delivery"]
    actual = values["actual_delivery"]
    label = (
        "delayed" if datetime.fromisoformat(actual) >
                     datetime.fromisoformat(scheduled)
        else "on_time"
    )
    b_template = info["task_B_templates"]["train"][position % 4]
    b = {
        "sample_id": f"{load_id}_B",
        **common,
        "task": "B",
        "messages": [
            system,
            {"role": "user", "content":
                info["task_B_instruction"] +
                b_template.format(scheduled=scheduled, actual=actual)},
            {"role": "assistant", "content":
                json.dumps({"delay_label": label}, ensure_ascii=False)},
        ],
        "template_id": f"train_B_{position % 4 + 1}",
    }
    return a, b

# 新增的 3,200 条业务记录中，固定 25% 做 Task A 主动缺失
new_ids = list(candidate_ids[:3200])
omit_rng = random.Random(5043)
omit_ids = set(omit_rng.sample(new_ids, 800))
omittable = [key for key in fields if key != "load_id"]

new_pairs = {}
for position, load_id in enumerate(new_ids):
    omitted = [omit_rng.choice(omittable)] if load_id in omit_ids else []
    new_pairs[load_id] = make_pair(load_id, position, omitted)

# E1 原样保留；E2 新增 1,200 对，E3 再新增 2,000 对
e2_samples = e1_samples + [
    sample for load_id in new_ids[:1200] for sample in new_pairs[load_id]
]
e3_samples = e1_samples + [
    sample for load_id in new_ids for sample in new_pairs[load_id]
]

print("E1：", len(e1_samples), "条")
print("E2：", len(e2_samples), "条")
print("E3：", len(e3_samples), "条")
print("E2 新增：", len(e2_samples) - len(e1_samples), "条")
print("E3 相比 E2 新增：", len(e3_samples) - len(e2_samples), "条")

### 1.9 校验扩展训练指令

检查 E2/E3 的数量、A/B 配对、样本编号和 Train 归属。
对新增指令逐条核对 Task A 的 JSON 字段与缺失值、
Task B 的时间标签；确认 E1 原样包含在 E2、E3 中。

In [ ]:
from collections import Counter
from datetime import datetime
import json

def check_dataset(name, samples, expected_count, expected_loads):
    ids = [row["sample_id"] for row in samples]
    tasks = Counter(row["task"] for row in samples)
    load_tasks = {}

    for row in samples:
        load_id = row["source_load_id"]
        assert split_by_load.get(load_id) == "train", row["sample_id"]
        load_tasks.setdefault(load_id, set()).add(row["task"])

        # 每条指令必须有 system、user、assistant 三段
        assert [m["role"] for m in row["messages"]] == [
            "system", "user", "assistant"
        ], row["sample_id"]

    assert len(samples) == expected_count
    assert len(ids) == len(set(ids)), f"{name} 存在重复 sample_id"
    assert tasks == {"A": expected_count // 2, "B": expected_count // 2}
    assert len(load_tasks) == expected_loads
    assert all(pair == {"A", "B"} for pair in load_tasks.values())

    print(f"{name}：{len(samples)} 条；A/B={dict(tasks)}；"
          f"业务记录 {len(load_tasks)} 条；编号、配对、划分通过")

check_dataset("E1", e1_samples, 1600, 800)
check_dataset("E2", e2_samples, 4000, 2000)
check_dataset("E3", e3_samples, 8000, 4000)

e1_sample_ids = {row["sample_id"] for row in e1_samples}
e2_sample_ids = {row["sample_id"] for row in e2_samples}
e3_sample_ids = {row["sample_id"] for row in e3_samples}
assert e1_sample_ids < e2_sample_ids < e3_sample_ids

# 逐条检查新生成的答案
for load_id, (a, b) in new_pairs.items():
    source = records_by_id.loc[load_id]

    a_answer = json.loads(a["messages"][2]["content"])
    assert set(a_answer) == set(fields), a["sample_id"]
    assert set(a["shown_fields"]).isdisjoint(a["omitted_fields"])
    assert all(a_answer[key] is None for key in fields
               if key not in a["shown_fields"]), a["sample_id"]

    scheduled = datetime.fromisoformat(str(source["scheduled_datetime"]))
    actual = datetime.fromisoformat(str(source["actual_datetime"]))
    expected_label = "delayed" if actual > scheduled else "on_time"
    b_answer = json.loads(b["messages"][2]["content"])
    assert b_answer == {"delay_label": expected_label}, b["sample_id"]

print("新增 Task A JSON/空值与 Task B 时间标签检查通过")
print("包含关系检查通过：E1 ⊂ E2 ⊂ E3")

### 1.10 核对字段值并保存训练数据

核对新增 Task A 的非空答案与原始业务记录一致。
保存 E2、E3 训练文件及固定 Val 文件，记录文件哈希。
Test 继续保存在 Phase 4 原数据包中。

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from collections import Counter
import hashlib
import json
import pandas as pd
from google.colab import files

# 核对新增 Task A 的每个实际展示值
for load_id, (a, b) in new_pairs.items():
    source = records_by_id.loc[load_id]
    expected = {
        "load_id": load_id,
        "truck_id": None if pd.isna(source["truck_id"]) else str(source["truck_id"]),
        "origin_city": str(source["origin_city"]),
        "destination_city": str(source["destination_city"]),
        "revenue_usd": float(source["revenue"]),
        "fuel_used_gallons": float(source["fuel_gallons_used"]),
        "scheduled_delivery": str(source["scheduled_datetime"]),
        "actual_delivery": str(source["actual_datetime"]),
    }
    answer = json.loads(a["messages"][2]["content"])
    user_text = a["messages"][1]["content"]

    for key in fields:
        correct = expected[key] if key in a["shown_fields"] else None
        assert answer[key] == correct, (a["sample_id"], key)
        if key in a["shown_fields"]:
            assert f"{labels[key]}：{expected[key]}" in user_text, (
                a["sample_id"], key
            )

print("新增 Task A 字段值、输入文本与答案逐项核对通过")

# 使用稳定的 JSONL 写法；E1 样本内容保持原样
def to_jsonl(rows):
    return "".join(
        json.dumps(row, ensure_ascii=False) + "\n" for row in rows
    ).encode("utf-8")

e2_bytes = to_jsonl(e2_samples)
e3_bytes = to_jsonl(e3_samples)

# 固定 Val 从原 Phase 4 包原样复制；不读取 Test
with ZipFile(phase4_path) as original:
    val_bytes = original.read("fleetmind_phase4_evaluation/val.jsonl")

manifest = {
    "project": "FleetMind-LLM",
    "phase": "5.1",
    "source_zip_sha256": source_sha,
    "selection_seed": 5042,
    "omission_seed": 5043,
    "training_counts": {"E1": 1600, "E2": 4000, "E3": 8000},
    "validation_count": 1000,
    "files_sha256": {
        "E2_train.jsonl": hashlib.sha256(e2_bytes).hexdigest(),
        "E3_train.jsonl": hashlib.sha256(e3_bytes).hexdigest(),
        "val.jsonl": hashlib.sha256(val_bytes).hexdigest(),
    },
}

output_zip = "FleetMind_LLM_Phase5_Training_Data.zip"
with ZipFile(output_zip, "w", compression=ZIP_DEFLATED) as z:
    z.writestr("E2_train.jsonl", e2_bytes)
    z.writestr("E3_train.jsonl", e3_bytes)
    z.writestr("val.jsonl", val_bytes)
    z.writestr(
        "manifest.json",
        json.dumps(manifest, ensure_ascii=False, indent=2),
    )

print("已保存：", output_zip)
print("文件哈希：", json.dumps(manifest["files_sha256"], indent=2))
files.download(output_zip)

## 2. 训练数据规模实验

E2 和 E3 分别从 Qwen3-0.6B 基座模型初始化新的 Rank 16 LoRA。
先核对训练文件、固定验证文件与 GPU 环境，再开始训练。
两组保持相同训练配置；训练步数会随数据量增加而变化。

In [ ]:
from google.colab import files

uploaded = files.upload()  # 选择桌面上的 FleetMind_LLM_Phase5_Training_Data.zip
print("已上传：", list(uploaded))

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import json
import torch

package = Path("FleetMind_LLM_Phase5_Training_Data.zip")
assert package.is_file(), "当前 Colab 会话找不到 Phase 5 数据包"

data_dir = Path("/content/fleetmind_phase5_data")
data_dir.mkdir(parents=True, exist_ok=True)

with ZipFile(package) as z:
    manifest = json.loads(z.read("manifest.json"))

    for name, expected_hash in manifest["files_sha256"].items():
        content = z.read(name)
        actual_hash = hashlib.sha256(content).hexdigest()
        assert actual_hash == expected_hash, f"{name} 哈希不一致"
        (data_dir / name).write_bytes(content)

print("数据文件：")
for name in ("E2_train.jsonl", "E3_train.jsonl", "val.jsonl"):
    path = data_dir / name
    count = sum(1 for _ in path.open(encoding="utf-8"))
    print(f"{path}: {count} 条")

print("\nPyTorch：", torch.__version__)
print("CUDA 可用：", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU：", torch.cuda.get_device_name(0))

### 2.2 对齐 E1 的训练口径

复核 Phase 3 的实际训练代码：E1 将提示部分的 labels 设为 -100，只对助手答案及结束标记计算 Loss。
E2、E3 把完整对话作为 text，并设置 assistant_only_loss=False、completion_only_loss=False，对完整序列计算 Loss。
因此三者的 Loss 数值不可直接比较；生成评测结果单独报告。

### 2.2 准备训练文本并检查长度

使用 Qwen 聊天模板将 E2、E3 和固定 Val 转为 text 格式。
关闭 thinking，分批统计 token 长度，检查是否超过 512。

In [ ]:
import json
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

def load_text_dataset(filename):
    # 读取已经解压到 data_dir 的 JSONL
    with (data_dir / filename).open(encoding="utf-8") as f:
        rows = [json.loads(line) for line in f]

    # 沿用 E1 的 Qwen 聊天模板，生成完整问答文本
    texts = [
        tokenizer.apply_chat_template(
            row["messages"],
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
        for row in rows
    ]
    return Dataset.from_dict({"text": texts})

e2_dataset = load_text_dataset("E2_train.jsonl")
e3_dataset = load_text_dataset("E3_train.jsonl")
val_dataset = load_text_dataset("val.jsonl")

def report_lengths(name, dataset, batch_size=256):
    lengths = []

    # 分批传入普通 Python list[str]，兼容当前 datasets 版本
    for start in range(0, len(dataset), batch_size):
        texts = [
            dataset[i]["text"]
            for i in range(start, min(start + batch_size, len(dataset)))
        ]
        assert all(isinstance(text, str) for text in texts)

        encoded = tokenizer(
            texts,
            add_special_tokens=False,
            truncation=False,
        )
        lengths.extend(len(ids) for ids in encoded["input_ids"])

    print(
        f"{name}: {len(lengths)} 条，"
        f"最长 {max(lengths)} tokens，"
        f"超过 512 的有 {sum(n > 512 for n in lengths)} 条"
    )
    return lengths

e2_lengths = report_lengths("E2 Train", e2_dataset)
e3_lengths = report_lengths("E3 Train", e3_dataset)
val_lengths = report_lengths("固定 Val", val_dataset)

print("\n首条文本开头：", repr(e2_dataset[0]["text"][:180]))

### 2.3 配置 E2 模型与训练器

从 Qwen3-0.6B 基座模型初始化新的 Rank 16 LoRA。
沿用 E1 的训练参数和完整 text 序列 Loss 口径，
使用固定 Val 1,000 选择 E2 的最佳 checkpoint。
本格只创建训练器，不启动训练。

In [ ]:
%pip install "trl==1.17.1"

In [ ]:
import trl
print("TRL 版本：", trl.__version__)

In [ ]:
%pip uninstall -y torchao

In [ ]:
import importlib.metadata
import torch
import gc

try:
    print("torchao 版本：", importlib.metadata.version("torchao"))
except importlib.metadata.PackageNotFoundError:
    print("torchao 已卸载")

# 清理上次创建训练器失败后留在内存中的模型
if "base_model" in globals():
    del base_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import torch
from transformers import AutoModelForCausalLM, set_seed
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

assert torch.cuda.is_available(), "请先在 Colab 选择 GPU 运行时"
set_seed(42)

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B",
    dtype=torch.float16,
)
base_model.config.use_cache = False

e2_lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

e2_args = SFTConfig(
    output_dir="/content/fleetmind_phase5_E2",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type="linear",
    warmup_steps=30,
    weight_decay=0.01,
    max_grad_norm=1.0,
    max_length=512,
    dataset_text_field="text",
    packing=False,
    assistant_only_loss=False,
    completion_only_loss=False,
    fp16=True,
    gradient_checkpointing=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=42,
    data_seed=42,
    report_to="none",
)

e2_trainer = SFTTrainer(
    model=base_model,
    args=e2_args,
    train_dataset=e2_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    peft_config=e2_lora,
)

print("E2 训练器已创建；训练尚未开始")
print("训练样本：", len(e2_trainer.train_dataset))
print("验证样本：", len(e2_trainer.eval_dataset))
print("LoRA Rank：", e2_lora.r)
print("预计每轮更新步数约：", 4000 // (2 * 8))

### 2.4 训练 E2 并保存结果

使用 4,000 条训练指令和固定 Val 1,000 训练 3 个 Epoch。
每轮验证并保存 checkpoint，训练结束后载入 Val Loss 最低的模型。
记录训练指标、最佳轮次和日志，导出 E2 adapter。

### E2 训练次数说明

E2 有 4,000 条训练指令。训练 **3 Epoch**，就是让模型把这 4,000 条指令完整学习 3 遍。每一遍结束后，再用固定的 1,000 条 Validation 检查效果。

训练时，模型不会看完全部 4,000 条才更新一次：

| 设置 | 当前数值 | 通俗理解 |
|---|---:|---|
| Batch size | 2 | 每次先看 2 条指令，计算它们带来的修改方向（梯度） |
| 梯度累积步数 | 8 | 累计 8 次修改方向后，才真正更新一次 |
| 每次更新涉及的样本 | 2 × 8 = 16 | 大约看过 16 条指令，更新一次 |
| 每轮更新次数 | 4,000 ÷ 16 = 250 | 走完一遍训练集，约更新 250 次 |
| 3 轮更新次数 | 250 × 3 = 750 | 整个 E2 训练约更新 750 次 |

这里的 **250 是“更新次数”，不是“参数个数”**。参数是模型内部可以调整的数字；一次更新会同时调整许多参数。

本项目使用 LoRA 微调：Qwen 基座模型的原有权重保持冻结，训练主要调整新增的 LoRA 参数。每次更新时，优化器根据前面累计的梯度，修改这些 LoRA 参数。

Validation 的 1,000 条指令用于观察 Loss 和选择最佳 checkpoint，不用于更新参数。最终仍需独立生成回答，计算 Task A 字段准确率和 Task B 分类指标。

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import json
from google.colab import files

# 正式训练 E2：3 个 Epoch
e2_result = e2_trainer.train()

# 确认训练和验证完成，防止再次保存半成品
assert e2_trainer.state.epoch >= 3.0
assert e2_trainer.state.best_metric is not None

# 保存验证 Loss 最低的 adapter
best_dir = Path("/content/fleetmind_phase5_E2_best_adapter")
e2_trainer.save_model(str(best_dir))
tokenizer.save_pretrained(str(best_dir))
e2_trainer.save_state()

# 从训练日志中记录 Loss 和 token Accuracy
metric_logs = [
    {
        "epoch": log.get("epoch"),
        "step": log.get("step"),
        "train_loss": log.get("loss"),
        "train_token_accuracy": log.get("mean_token_accuracy"),
        "val_loss": log.get("eval_loss"),
        "val_token_accuracy": log.get("eval_mean_token_accuracy"),
    }
    for log in e2_trainer.state.log_history
    if "loss" in log or "eval_loss" in log
]

summary = {
    "experiment": "E2",
    "train_samples": 4000,
    "validation_samples": 1000,
    "lora_rank": 16,
    "train_metrics": e2_result.metrics,
    "metric_logs": metric_logs,
    "best_model_checkpoint": e2_trainer.state.best_model_checkpoint,
    "best_validation_loss": e2_trainer.state.best_metric,
    "completed_optimizer_steps": e2_trainer.state.global_step,
    "completed_epochs": e2_trainer.state.epoch,
}

summary_path = Path("/content/fleetmind_phase5_E2_summary.json")
summary_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# 打包模型和日志
result_zip = Path("/content/FleetMind_LLM_Phase5_E2_Results.zip")
with ZipFile(result_zip, "w", ZIP_DEFLATED) as z:
    for path in best_dir.rglob("*"):
        if path.is_file():
            z.write(path, arcname=f"best_adapter/{path.relative_to(best_dir)}")

    state_path = Path(e2_args.output_dir) / "trainer_state.json"
    if state_path.is_file():
        z.write(state_path, arcname="trainer_state.json")

    z.write(summary_path, arcname="summary.json")

print("E2 完成步数：", summary["completed_optimizer_steps"])
print("最佳 Val Loss：", summary["best_validation_loss"])
print("每轮 Val 指标：")
for log in metric_logs:
    if log["val_loss"] is not None:
        print(log)

files.download(str(result_zip))

### 2.5 配置 E3 训练器

释放 E2 模型显存，从 Qwen3-0.6B 基座重新初始化 Rank 16 LoRA。
沿用 E2 的训练配置，改用 8,000 条训练指令；
固定 Val 1,000 不变。本格只创建训练器。

In [ ]:
from dataclasses import replace
import gc
import torch
from transformers import AutoModelForCausalLM, set_seed
from peft import LoraConfig
from trl import SFTTrainer

# E2 结果已保存；释放它占用的 GPU 显存
del e2_trainer
del base_model
gc.collect()
torch.cuda.empty_cache()

# 重新设置种子，从原始 Qwen 基座加载，而不是接着 E2 训练
set_seed(42)
e3_base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B",
    dtype=torch.float16,
)
e3_base_model.config.use_cache = False

# 新建与 E2 相同的 LoRA 配置
e3_lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

# 复制 E2 的训练设置，只更换输出目录
e3_args = replace(
    e2_args,
    output_dir="/content/fleetmind_phase5_E3",
)

e3_trainer = SFTTrainer(
    model=e3_base_model,
    args=e3_args,
    train_dataset=e3_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    peft_config=e3_lora,
)

print("E3 训练器已创建；训练尚未开始")
print("训练样本：", len(e3_trainer.train_dataset))
print("验证样本：", len(e3_trainer.eval_dataset))
print("LoRA Rank：", e3_lora.r)
print("初始更新步数：", e3_trainer.state.global_step)
print("预计 3 轮更新步数约：", 3 * 8000 // (2 * 8))

##2.6 训练 E3
E3 使用 8,000 条训练指令，训练 3 轮，预计更新 1,500 步。每轮会在同一份 Val 1,000 上记录验证损失和 token accuracy。训练完成后再打包下载结果。

In [ ]:
# 确认 E3 是全新训练，避免误把中途运行当作已完成
assert e3_trainer.state.global_step == 0, "E3 已有训练步数，请先检查是否需要从 checkpoint 恢复"

# 开始训练；配置会在每轮结束时验证，并载入 Val Loss 最低的 checkpoint
e3_train_result = e3_trainer.train()

# 只有完整训练到 1500 步，才报告为 E3 已完成
assert e3_trainer.state.global_step == 1500, (
    f"E3 仅完成 {e3_trainer.state.global_step}/1500 步，请检查训练日志"
)

print("===== E3 训练结果 =====")
print("完成步数：", e3_trainer.state.global_step)
print("最佳 checkpoint：", e3_trainer.state.best_model_checkpoint)
print("最佳 Val Loss：", e3_trainer.state.best_metric)

# 提取每轮验证指标；token accuracy 是预测下一个 token 的准确率，不是 Task A/B 的业务准确率
for row in e3_trainer.state.log_history:
    if "eval_loss" in row:
        print({
            "epoch": row.get("epoch"),
            "step": row.get("step"),
            "val_loss": row["eval_loss"],
            "val_token_accuracy": row.get("eval_mean_token_accuracy"),
        })

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import json
import shutil
from google.colab import files

# 先确认训练完整，且最佳 checkpoint 已确定
assert e3_trainer.state.global_step == 1500, "E3 尚未完成 1500 步"
assert e3_trainer.args.load_best_model_at_end, "训练配置未启用载入最佳模型"
assert e3_trainer.state.best_model_checkpoint is not None, "未找到最佳 checkpoint"

result_dir = Path("/content/FleetMind_LLM_Phase5_E3_Results")
adapter_dir = result_dir / "best_adapter"
result_dir.mkdir(parents=True, exist_ok=True)

# 训练结束时，Trainer 已将最佳 checkpoint 的权重载入模型
e3_trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
e3_trainer.state.save_to_json(str(result_dir / "trainer_state.json"))

# 单独保存便于阅读的训练摘要
val_history = [
    {
        "epoch": row.get("epoch"),
        "step": row.get("step"),
        "val_loss": row["eval_loss"],
        "val_token_accuracy": row.get("eval_mean_token_accuracy"),
    }
    for row in e3_trainer.state.log_history
    if "eval_loss" in row
]
summary = {
    "experiment": "E3",
    "base_model": "Qwen/Qwen3-0.6B",
    "train_samples": 8000,
    "validation_samples": 1000,
    "lora_rank": 16,
    "epochs": 3,
    "completed_steps": e3_trainer.state.global_step,
    "best_checkpoint": e3_trainer.state.best_model_checkpoint,
    "best_val_loss": e3_trainer.state.best_metric,
    "validation_history": val_history,
}
(result_dir / "training_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# 检查 adapter 权重确实已保存
assert (adapter_dir / "adapter_config.json").is_file()
assert any(adapter_dir.glob("adapter_model.*")), "未找到 adapter 权重文件"

# 打包结果，不重复打包训练过程中的 checkpoint
zip_path = Path("/content/FleetMind_LLM_Phase5_E3_Results.zip")
with ZipFile(zip_path, "w", ZIP_DEFLATED) as archive:
    for path in result_dir.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(result_dir.parent))

# 如果 Google Drive 已挂载，额外保存一份，避免 Colab 会话结束后丢失
drive_dir = Path("/content/drive/MyDrive")
if drive_dir.is_dir():
    shutil.copy2(zip_path, drive_dir / zip_path.name)
    print("Drive 备份：", drive_dir / zip_path.name)

print("E3 结果包：", zip_path)
print("文件大小：", round(zip_path.stat().st_size / 1024**2, 2), "MB")
files.download(str(zip_path))

### 3.1 准备统一验证评测

E1、E2、E3 已分别使用 1,600、4,000、8,000 条训练指令完成训练。现在要让三个模型回答**同一份 Validation 题目**，观察增加训练数据后，实际任务表现有没有改善。

本步骤先准备评测材料：

1. 读取三个实验保存的最佳 LoRA adapter，并核对 Rank 都是 16。
2. 核对固定 Validation：共 1,000 条，其中 Task A 信息提取 500 条、Task B 延误判断 500 条。
3. 检查文件是否完整，避免后续生成时因缺少模型文件或用错数据而中断。

这一步**只检查和准备文件，不运行模型预测**。后续会先用少量样本确认输出格式，再对全部 1,000 条进行生成评测。最终模型只根据 Validation 结果选择；封存的 Test 暂不使用。

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from google.colab import files
import hashlib
import json
import shutil

content_dir = Path("/content")
eval_dir = content_dir / "fleetmind_phase5_eval"
eval_dir.mkdir(exist_ok=True)

def find_or_upload(prefix):
    # 优先使用当前 Colab 已有的压缩包
    matches = list(content_dir.glob(f"{prefix}*.zip"))
    if not matches:
        print(f"请上传 {prefix}.zip")
        files.upload()
        matches = list(content_dir.glob(f"{prefix}*.zip"))
    assert matches, f"仍未找到 {prefix} 的压缩包"
    return max(matches, key=lambda p: p.stat().st_mtime)

packages = {
    "E1": find_or_upload("FleetMind_LLM_Phase3_Results"),
    "E2": find_or_upload("FleetMind_LLM_Phase5_E2_Results"),
    "E3": find_or_upload("FleetMind_LLM_Phase5_E3_Results"),
}

for name, package in packages.items():
    adapter_dir = eval_dir / name / "best_adapter"
    adapter_dir.mkdir(parents=True, exist_ok=True)

    with ZipFile(package) as archive:
        assert archive.testzip() is None, f"{name} 压缩包校验失败"

        # 三个结果包的外层目录可能不同，统一取其中的 best_adapter
        members = [
            item for item in archive.infolist()
            if "/best_adapter/" in f"/{item.filename}" and not item.is_dir()
        ]
        assert members, f"{name} 没有找到 best_adapter"

        for item in members:
            relative_name = item.filename.split("best_adapter/", 1)[1]
            target = adapter_dir / relative_name
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(item) as source, target.open("wb") as output:
                shutil.copyfileobj(source, output)

    config = json.loads((adapter_dir / "adapter_config.json").read_text())
    assert config["r"] == 16, f"{name} 的 Rank 不是 16"
    assert (adapter_dir / "adapter_model.safetensors").is_file()
    print(f"{name}：adapter 已准备；Rank={config['r']}")

# 使用 Phase 5 数据包中的固定 Val；缺失时再上传数据包
val_path = content_dir / "fleetmind_phase5_data" / "val.jsonl"
if not val_path.is_file():
    data_package = find_or_upload("FleetMind_LLM_Phase5_Training_Data")
    val_path.parent.mkdir(parents=True, exist_ok=True)
    with ZipFile(data_package) as archive:
        val_path.write_bytes(archive.read("val.jsonl"))

val_hash = hashlib.sha256(val_path.read_bytes()).hexdigest()
assert val_hash == "8223dffee4c4fe557a8a29d116bb3be048b74578b9e01d7ea109ef732f243b4c"

with val_path.open(encoding="utf-8") as file:
    val_rows = [json.loads(line) for line in file]

assert len(val_rows) == 1000
assert sum(row["task"] == "A" for row in val_rows) == 500
assert sum(row["task"] == "B" for row in val_rows) == 500
print("固定 Val 核对通过：1,000 条；Task A/B 各 500 条")

### 3.2 少量样本生成检查
先让每个模型回答同样的 1 条 Task A + 1 条 Task B。这一步只检查能否正常生成 JSON，以及输入是否正确排除了标准答案；不把这 6 次输出当作正式成绩。

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 清理上次报错后可能留在显存中的模型
for name in ("model", "base", "inputs", "output"):
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()

# 三个模型回答相同的两道题：Task A、Task B 各一条
trial_rows = [
    next(row for row in val_rows if row["task"] == "A"),
    next(row for row in val_rows if row["task"] == "B"),
]
eval_tokenizer = AutoTokenizer.from_pretrained(
    str(eval_dir / "E1" / "best_adapter")
)

for experiment in ("E1", "E2", "E3"):
    print(f"\n===== {experiment} 试生成 =====")

    # 重新加载同一个基座模型，挂载当前实验的 LoRA adapter
    base = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3-0.6B", dtype=torch.float16
    ).to("cuda")
    model = PeftModel.from_pretrained(
        base, str(eval_dir / experiment / "best_adapter")
    ).eval()

    for row in trial_rows:
        # 只输入 system 和 user，不把 assistant 标准答案给模型
        inputs = eval_tokenizer.apply_chat_template(
            row["messages"][:-1],
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=False,
            return_dict=True,
            return_tensors="pt",
        ).to("cuda")

        with torch.inference_mode():
            output = model.generate(
                **inputs,  # 展开 input_ids、attention_mask 等字段
                max_new_tokens=192,
                do_sample=False,
                pad_token_id=eval_tokenizer.pad_token_id,
                eos_token_id=eval_tokenizer.eos_token_id,
            )

        # 只解码模型新生成的部分
        prompt_length = inputs["input_ids"].shape[-1]
        prediction = eval_tokenizer.decode(
            output[0, prompt_length:],
            skip_special_tokens=True,
        ).strip()

        print("样本：", row["sample_id"])
        print("模型回答：", prediction)
        print("标准答案：", row["messages"][-1]["content"])

    del model, base, inputs, output
    gc.collect()
    torch.cuda.empty_cache()

### 3.3 开始完整 Val 生成：先跑 E1
先保存 E1 对 Val 1,000 条的逐题回答。代码按小批次生成，并持续写入文件；如果中断，重新运行会按样本编号接着做。E1 完成后再用同一函数运行 E2、E3。

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm
import gc
import json
import shutil
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

prediction_dir = eval_dir / "predictions"
prediction_dir.mkdir(exist_ok=True)
eval_tokenizer.padding_side = "left"  # 批量生成时，短题目在左侧补齐

def run_val_generation(experiment, batch_size=8):
    prediction_path = prediction_dir / f"{experiment}_val_predictions.jsonl"
    drive_dir = Path("/content/drive/MyDrive")

    # 若 Colab 本地文件丢失，尝试从已挂载的 Drive 恢复进度
    if not prediction_path.exists() and (drive_dir / prediction_path.name).is_file():
        shutil.copy2(drive_dir / prediction_path.name, prediction_path)

    completed = set()
    if prediction_path.exists():
        with prediction_path.open(encoding="utf-8") as file:
            for line in file:
                completed.add(json.loads(line)["sample_id"])

    expected_ids = {row["sample_id"] for row in val_rows}
    assert completed <= expected_ids, "已有预测文件包含不属于固定 Val 的样本"
    remaining = [row for row in val_rows if row["sample_id"] not in completed]
    print(f"{experiment}：已有 {len(completed)} 条，待生成 {len(remaining)} 条")

    if not remaining:
        assert len(completed) == 1000
        print(f"{experiment} 的 Val 1,000 条已全部生成：{prediction_path}")
        return prediction_path

    # 每个实验都从相同的基座模型加载自己的 adapter
    base = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3-0.6B", dtype=torch.float16
    ).to("cuda")
    model = PeftModel.from_pretrained(
        base, str(eval_dir / experiment / "best_adapter")
    ).eval()

    try:
        # A、B 分开成批：B 的答案更短，可以少生成一些 token
        for task, max_tokens in (("A", 192), ("B", 48)):
            task_rows = [row for row in remaining if row["task"] == task]

            for start in tqdm(
                range(0, len(task_rows), batch_size),
                desc=f"{experiment} Task {task}",
            ):
                batch = task_rows[start:start + batch_size]

                # 每道题只传入 system 和 user，排除 assistant 标准答案
                questions = [row["messages"][:-1] for row in batch]
                inputs = eval_tokenizer.apply_chat_template(
                    questions,
                    tokenize=True,
                    add_generation_prompt=True,
                    enable_thinking=False,
                    padding=True,
                    return_dict=True,
                    return_tensors="pt",
                ).to("cuda")

                with torch.inference_mode():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=max_tokens,
                        do_sample=False,
                        pad_token_id=eval_tokenizer.pad_token_id,
                        eos_token_id=eval_tokenizer.eos_token_id,
                    )

                prompt_length = inputs["input_ids"].shape[-1]
                answers = eval_tokenizer.batch_decode(
                    outputs[:, prompt_length:],
                    skip_special_tokens=True,
                )

                # 每批立即写盘，便于中断后按 sample_id 接续
                with prediction_path.open("a", encoding="utf-8") as file:
                    for row, answer in zip(batch, answers):
                        record = {
                            "sample_id": row["sample_id"],
                            "task": row["task"],
                            "prediction": answer.strip(),
                            "reference": row["messages"][-1]["content"],
                        }
                        file.write(json.dumps(record, ensure_ascii=False) + "\n")

                # 已挂载 Drive 时，同步备份当前进度
                if drive_dir.is_dir():
                    shutil.copy2(prediction_path, drive_dir / prediction_path.name)

        # 只有数量和编号全部吻合，才报告完成
        with prediction_path.open(encoding="utf-8") as file:
            ids = [json.loads(line)["sample_id"] for line in file]
        assert len(ids) == 1000 and set(ids) == expected_ids
        print(f"{experiment} 完成：1,000/1,000 条")
        print("预测文件：", prediction_path)
        return prediction_path

    finally:
        del model, base
        gc.collect()
        torch.cuda.empty_cache()

# 本次先运行 E1
e1_prediction_path = run_val_generation("E1")

In [ ]:
from google.colab import files

files.download(str(e1_prediction_path))

### 3.3 E2 完整验证集生成

使用 E2 的最佳 LoRA adapter，回答与 E1 相同的 Val 1,000 条题目，并保存逐题预测。本步骤只生成答案，暂不评分。

In [ ]:
# 使用与 E1 完全相同的生成函数和验证集
e2_prediction_path = run_val_generation("E2")

In [ ]:
from google.colab import files

# 先保存已完成的 E2 预测
files.download(str(e2_prediction_path))

### 3.3 E3 完整验证集生成

使用 E3 的最佳 LoRA adapter，回答同一份 Val 1,000 条题目。完成后，三个模型才都有可用于正式评分的逐题预测。

In [ ]:
# 使用相同题目和生成设置运行 E3
e3_prediction_path = run_val_generation("E3")

In [ ]:
# 单独下载 E3 的逐题预测
from google.colab import files

files.download(str(e3_prediction_path))

### 3.4 固定 Validation 统一评分

使用同一套评分规则比较 E1、E2、E3。Task A 统计 JSON 有效率、结构准确率、8 个字段的准确率和空值准确率；Task B 统计 Accuracy、两类 F1，以及重点关注的 `on_time` Recall。

评分前会核对：每个模型都有 Val 的全部 1,000 个样本编号，且文件中的标准答案与固定 Val 一致。本步骤只评估 Validation，不使用 Test。

In [ ]:
# 5.3.4：对 E1、E2、E3 的固定 Val 预测统一评分
import json

expected = {row["sample_id"]: row for row in val_rows}
field_names = [
    "load_id", "truck_id", "origin_city", "destination_city",
    "revenue_usd", "fuel_used_gallons",
    "scheduled_delivery", "actual_delivery",
]

def class_scores(gold, predicted, label):
    # 分别计算一个类别的 Precision、Recall 和 F1
    tp = sum(g == label and p == label for g, p in zip(gold, predicted))
    fp = sum(g != label and p == label for g, p in zip(gold, predicted))
    fn = sum(g == label and p != label for g, p in zip(gold, predicted))
    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    return {"precision": precision, "recall": recall, "f1": f1}

def score_experiment(experiment):
    path = prediction_dir / f"{experiment}_val_predictions.jsonl"
    with path.open(encoding="utf-8") as file:
        rows = [json.loads(line) for line in file]

    # 确认恰好覆盖固定 Val 的全部 1,000 道题
    ids = [row["sample_id"] for row in rows]
    assert len(ids) == len(set(ids)) == 1000
    assert set(ids) == set(expected)

    a_count = a_valid = a_structure = a_exact = a_fields = 0
    null_correct = null_total = 0
    b_gold, b_predicted = [], []

    for row in rows:
        source = expected[row["sample_id"]]
        assert row["task"] == source["task"]
        gold = json.loads(source["messages"][-1]["content"])
        assert json.loads(row["reference"]) == gold

        try:
            predicted = json.loads(row["prediction"])
            valid_json = True
        except (json.JSONDecodeError, TypeError):
            predicted = None
            valid_json = False

        if row["task"] == "A":
            a_count += 1
            a_valid += valid_json
            correct_shape = isinstance(predicted, dict) and set(predicted) == set(field_names)
            a_structure += correct_shape
            a_exact += correct_shape and predicted == gold

            # 每条 Task A 有 8 个字段，总计 4,000 个字段
            for field in field_names:
                correct = (
                    isinstance(predicted, dict)
                    and field in predicted
                    and predicted[field] == gold[field]
                )
                a_fields += correct
                if gold[field] is None:
                    null_total += 1
                    null_correct += correct
        else:
            b_gold.append(gold["delay_label"])

            # 无效 JSON、字段不对或标签不对，都算作错误预测
            label = (
                predicted["delay_label"]
                if isinstance(predicted, dict)
                and set(predicted) == {"delay_label"}
                and predicted["delay_label"] in {"delayed", "on_time"}
                else "__invalid__"
            )
            b_predicted.append(label)

    assert a_count == len(b_gold) == 500
    delayed = class_scores(b_gold, b_predicted, "delayed")
    on_time = class_scores(b_gold, b_predicted, "on_time")

    return {
        "experiment": experiment,
        "A_json_valid": f"{a_valid}/500",
        "A_structure": f"{a_structure}/500",
        "A_field_accuracy": f"{a_fields}/4000",
        "A_null_accuracy": f"{null_correct}/{null_total}",
        "A_exact_records": f"{a_exact}/500",
        "B_accuracy": f"{sum(g == p for g, p in zip(b_gold, b_predicted))}/500",
        "B_delayed_F1": round(delayed["f1"], 4),
        "B_on_time_F1": round(on_time["f1"], 4),
        "B_macro_F1": round((delayed["f1"] + on_time["f1"]) / 2, 4),
        "B_on_time_recall": round(on_time["recall"], 4),
        "B_invalid_outputs": b_predicted.count("__invalid__"),
    }

comparison = [score_experiment(name) for name in ("E1", "E2", "E3")]

print("===== 同一份 Val 1,000 的生成评测 =====")
for result in comparison:
    print(json.dumps(result, ensure_ascii=False))

# 保存评分表，供下一步分析和选模型
metrics_path = eval_dir / "E1_E2_E3_val_metrics.json"
metrics_path.write_text(
    json.dumps(comparison, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("指标文件：", metrics_path)

###3.5 Task B 错题与题目表述分析

比较三个模型在同一道题上的错误，并按 Validation 的题目模板统计。重点检查 E3 的新增错误是否集中在某一种时间描述方式，以及 E1、E2 是否真正答对了相同的题目。

In [ ]:
from collections import Counter
import json

b_rows = {name: {} for name in ("E1", "E2", "E3")}

for name in b_rows:
    path = prediction_dir / f"{name}_val_predictions.jsonl"
    with path.open(encoding="utf-8") as file:
        for line in file:
            row = json.loads(line)
            if row["task"] == "B":
                b_rows[name][row["sample_id"]] = row

for name, rows in b_rows.items():
    errors = []
    by_template = Counter()

    for sample_id, row in rows.items():
        template = expected[sample_id]["template_id"]
        gold = json.loads(row["reference"])["delay_label"]
        predicted = json.loads(row["prediction"])["delay_label"]

        by_template[(template, "总数")] += 1
        if predicted != gold:
            by_template[(template, "错误")] += 1
            errors.append(sample_id)

    print(f"\n{name}：Task B 错误 {len(errors)}/500")
    print("错误编号：", errors[:10])
    print("按题目表述统计：", dict(by_template))

# 展示 E3 中「真实延误却回答准时」的前三道题
print("\n===== E3 代表性新增错误 =====")
shown = 0
for sample_id, row in b_rows["E3"].items():
    gold = json.loads(row["reference"])["delay_label"]
    predicted = json.loads(row["prediction"])["delay_label"]
    if gold == "delayed" and predicted == "on_time":
        print("\n编号：", sample_id)
        print("题目：", expected[sample_id]["messages"][1]["content"])
        print("E3 回答：", row["prediction"])
        shown += 1
        if shown == 3:
            break

###错题分析说明：
E1、E2 各错 3 题，但并非完全相同的 3 题；E3 的退步主要集中在 validation_B_2 这种“先写实际时间、再写计划时间”的表述。代表案例相差数小时，因此不能用微秒边界错误解释 E3 的下降。我们能确认的是表述相关的性能退化，目前不能断言训练数据规模就是唯一原因

## 3.6 查看 E2/E3 的 Loss 曲线
E1 原训练使用的 Val 规模不同，这张图只画 E2、E3：左边看训练 Loss 的变化，右边放大看每轮 Val Loss。代码读取已保存的训练日志，不会重新训练。

In [ ]:
import json
import matplotlib.pyplot as plt
from zipfile import ZipFile

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for name in ("E2", "E3"):
    # 从结果包读取原始训练记录
    with ZipFile(packages[name]) as archive:
        state_file = next(
            path for path in archive.namelist()
            if path.endswith("trainer_state.json")
        )
        history = json.loads(archive.read(state_file))["log_history"]

    train_logs = [row for row in history if "loss" in row]
    val_logs = [row for row in history if "eval_loss" in row]

    axes[0].plot(
        [row["epoch"] for row in train_logs],
        [row["loss"] for row in train_logs],
        label=name,
    )
    axes[1].plot(
        [row["epoch"] for row in val_logs],
        [row["eval_loss"] for row in val_logs],
        marker="o",
        label=name,
    )

axes[0].set(title="Training Loss", xlabel="Epoch", ylabel="Loss")
axes[1].set(title="Validation Loss", xlabel="Epoch", ylabel="Loss")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
loss_plot_path = eval_dir / "E2_E3_loss_curves.png"
plt.savefig(loss_plot_path, dpi=160, bbox_inches="tight")
plt.show()
print("曲线已保存：", loss_plot_path)

## 固定 Validation 评测总结

三个模型分别使用 1,600、4,000、8,000 条训练指令，LoRA Rank 均为 16。我们让它们在**同一份 Val 1,000** 上按相同设置生成答案，并逐题评分；其中 Task A、Task B 各 500 条。

| 模型 | Task A 字段准确率 | Task A 空值准确率 | Task B Accuracy | Task B Macro-F1 |
| --- | ---: | ---: | ---: | ---: |
| E1（1,600 条） | 4000/4000 | 197/197 | 497/500（99.4%） | 0.9932 |
| E2（4,000 条） | 4000/4000 | 197/197 | 497/500（99.4%） | 0.9932 |
| E3（8,000 条） | 4000/4000 | 197/197 | 437/500（87.4%） | 0.8677 |

**主要发现**：三个模型的 Task A 均全部正确。E2 的 Task B 总体成绩与 E1 持平，但两者错题并不完全相同。E3 有 62 条真实延误被误判为准时，其中 61 条集中在 `validation_B_2` 这种时间表述；代表案例相差数小时，不能归为微秒边界问题。

E2、E3 的训练日志均显示第 1 轮 Val Loss 最低，之后上升。训练时的 token accuracy 和 Loss 只能辅助分析，最终仍以实际生成的 Task A/B 指标为准。E1 训练时使用旧 Val 200，因此它的训练期 Val Loss 不与 E2/E3 直接比较。

**阶段结论**：增加训练数据没有带来稳定提升，E3 在 Task B 上明显退步。E1 是下一步最终模型选择的优先候选；正式选择与锁定在 5.4 完成。封存的 Test 2,000 尚未生成预测或评分。

### 4.1 固定最终模型

根据同一份 Validation 1,000 的实际生成结果，选择 E1 作为最终模型。E1 与 E2 的 Task A 和 Task B 总体指标持平；E1 使用的训练数据更少。E3 的 Task B 表现下降，尤其集中在一种时间表述方式上。

从此固定 E1 adapter、生成设置和评分口径。下一步才对封存 Test 2,000 进行一次最终评测，不根据 Test 结果重新选模型。

In [ ]:
import hashlib
import json
from pathlib import Path

selected_name = "E1"
selected_adapter = eval_dir / selected_name / "best_adapter"

def sha256_file(path):
    # 记录文件指纹，方便以后确认使用的是同一份模型
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

selection = {
    "selected_experiment": selected_name,
    "base_model": "Qwen/Qwen3-0.6B",
    "adapter_dir": str(selected_adapter),
    "lora_rank": 16,
    "train_samples": 1600,
    "selection_split": "validation",
    "validation_samples": 1000,
    "validation_sha256": sha256_file(val_path),
    "generation": {
        "do_sample": False,
        "enable_thinking": False,
        "max_new_tokens_A": 192,
        "max_new_tokens_B": 48,
    },
    "adapter_config_sha256": sha256_file(
        selected_adapter / "adapter_config.json"
    ),
    "adapter_weights_sha256": sha256_file(
        selected_adapter / "adapter_model.safetensors"
    ),
    "validation_comparison": comparison,
    "selection_reason": (
        "E1 与 E2 的 Val 任务指标持平；E1 使用更少训练数据。"
        "E3 的 Task B Accuracy 降至 437/500。"
    ),
    "test_status": "尚未生成预测或评分",
}

selection_path = eval_dir / "final_model_selection.json"
selection_path.write_text(
    json.dumps(selection, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("最终选择：", selection["selected_experiment"])
print("选择记录：", selection_path)
print("Test 状态：", selection["test_status"])

4.2 核对封存的 Test

在最终模型 E1 和生成设置锁定后，读取 Phase 4 数据包中的 Test。核对文件哈希、2,000 条指令的 A/B 数量，以及 Test 与 Val 的业务记录不重叠。检查通过后，下一步才运行一次最终生成评测。

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from google.colab import files
import hashlib
import json

# 优先使用当前 Colab 已有的数据包；找不到才从电脑上传
packages_found = list(Path("/content").glob(
    "FleetMind_LLM_Phase4_Evaluation_Dataset*.zip"
))
if not packages_found:
    print("请上传 FleetMind_LLM_Phase4_Evaluation_Dataset.zip")
    files.upload()
    packages_found = list(Path("/content").glob(
        "FleetMind_LLM_Phase4_Evaluation_Dataset*.zip"
    ))
assert packages_found, "未找到 Phase 4 评测数据包"
test_package = max(packages_found, key=lambda p: p.stat().st_mtime)

# 仅取出封存的 test.jsonl
with ZipFile(test_package) as archive:
    assert archive.testzip() is None, "数据包损坏"
    test_member = next(
        name for name in archive.namelist()
        if name.endswith("/test.jsonl")
    )
    test_bytes = archive.read(test_member)

test_path = eval_dir / "test.jsonl"
test_path.write_bytes(test_bytes)

# 核对原始文件指纹与样本数量
test_hash = hashlib.sha256(test_bytes).hexdigest()
assert test_hash == (
    "700f21dc709f0eb40f1fcb323f0fa6ee675e7399c3e9b2fbd77f9f1cf1976473"
)
test_rows = [json.loads(line) for line in test_bytes.splitlines()]
assert len(test_rows) == 2000
assert sum(row["task"] == "A" for row in test_rows) == 1000
assert sum(row["task"] == "B" for row in test_rows) == 1000
assert all(row["split"] == "test" for row in test_rows)

# 再确认 Test 的编号和业务记录不属于 Val
assert not (
    {row["sample_id"] for row in test_rows}
    & {row["sample_id"] for row in val_rows}
)
assert not (
    {row["source_load_id"] for row in test_rows}
    & {row["source_load_id"] for row in val_rows}
)

# 确认最终模型仍是刚才锁定的 E1，权重没有改变
locked = json.loads(selection_path.read_text(encoding="utf-8"))
assert locked["selected_experiment"] == "E1"
assert locked["adapter_weights_sha256"] == sha256_file(
    selected_adapter / "adapter_model.safetensors"
)

print("最终模型：", locked["selected_experiment"])
print("Test 核对通过：2,000 条；Task A/B 各 1,000 条")
print("Test SHA256：", test_hash)
print("状态：尚未生成 Test 预测或评分")

### 4.3 最终 Test 生成

使用已根据 Validation 选定的 E1 adapter，按固定生成设置回答 Test 2,000 条题目（Task A/B 各 1,000 条）。逐批保存预测；如果 Colab 中断，可从已保存的样本继续。此后不根据 Test 表现更换模型或调整生成参数。

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm
import gc
import json
import shutil
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

# 再次确认模型、权重和生成参数与锁定记录一致
locked = json.loads(selection_path.read_text(encoding="utf-8"))
assert locked["selected_experiment"] == "E1"
assert locked["adapter_weights_sha256"] == sha256_file(
    selected_adapter / "adapter_model.safetensors"
)
assert locked["generation"] == {
    "do_sample": False,
    "enable_thinking": False,
    "max_new_tokens_A": 192,
    "max_new_tokens_B": 48,
}

test_prediction_path = prediction_dir / "E1_test_predictions.jsonl"
drive_dir = Path("/content/drive/MyDrive")
eval_tokenizer.padding_side = "left"

# Colab 本地文件丢失时，从 Drive 恢复已保存的进度
if not test_prediction_path.exists() and (drive_dir / test_prediction_path.name).is_file():
    shutil.copy2(drive_dir / test_prediction_path.name, test_prediction_path)

completed = set()
if test_prediction_path.exists():
    with test_prediction_path.open(encoding="utf-8") as file:
        for line in file:
            completed.add(json.loads(line)["sample_id"])

expected_test_ids = {row["sample_id"] for row in test_rows}
assert completed <= expected_test_ids
remaining = [row for row in test_rows if row["sample_id"] not in completed]
print(f"Test：已有 {len(completed)} 条，待生成 {len(remaining)} 条")

if remaining:
    # 只加载事先锁定的 E1
    base = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3-0.6B", dtype=torch.float16
    ).to("cuda")
    model = PeftModel.from_pretrained(base, str(selected_adapter)).eval()

    try:
        for task, max_tokens in (("A", 192), ("B", 48)):
            task_rows = [row for row in remaining if row["task"] == task]

            for start in tqdm(range(0, len(task_rows), 8), desc=f"Test Task {task}"):
                batch = task_rows[start:start + 8]

                # 只输入题目，不把 assistant 标准答案交给模型
                inputs = eval_tokenizer.apply_chat_template(
                    [row["messages"][:-1] for row in batch],
                    tokenize=True,
                    add_generation_prompt=True,
                    enable_thinking=False,
                    padding=True,
                    return_dict=True,
                    return_tensors="pt",
                ).to("cuda")

                with torch.inference_mode():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=max_tokens,
                        do_sample=False,
                        pad_token_id=eval_tokenizer.pad_token_id,
                        eos_token_id=eval_tokenizer.eos_token_id,
                    )

                prompt_length = inputs["input_ids"].shape[-1]
                answers = eval_tokenizer.batch_decode(
                    outputs[:, prompt_length:],
                    skip_special_tokens=True,
                )

                # 每批写入文件，并在 Drive 已挂载时备份
                with test_prediction_path.open("a", encoding="utf-8") as file:
                    for row, answer in zip(batch, answers):
                        record = {
                            "sample_id": row["sample_id"],
                            "task": row["task"],
                            "prediction": answer.strip(),
                            "reference": row["messages"][-1]["content"],
                        }
                        file.write(json.dumps(record, ensure_ascii=False) + "\n")

                if drive_dir.is_dir():
                    shutil.copy2(
                        test_prediction_path,
                        drive_dir / test_prediction_path.name,
                    )
    finally:
        del model, base
        gc.collect()
        torch.cuda.empty_cache()

# 只有 2,000 个编号齐全且无重复，才宣布生成完成
with test_prediction_path.open(encoding="utf-8") as file:
    generated_ids = [json.loads(line)["sample_id"] for line in file]

assert len(generated_ids) == len(set(generated_ids)) == 2000
assert set(generated_ids) == expected_test_ids
print("E1 Test 预测完成：2,000/2,000")
print("预测文件：", test_prediction_path)
print("状态：已生成预测，尚未评分")

### 保存最终 Test 预测

下载 E1 对 Test 2,000 条题目的逐题回答。这个文件是最终评分和错题复核的原始证据。

In [ ]:
from google.colab import files

# 单独下载最终 Test 的逐题预测
files.download(str(test_prediction_path))

### 4.5 最终 Test 评分

对已锁定的 E1 在 Test 2,000 上的逐题预测评分。Task A 统计 JSON、结构、字段和空值准确率；Task B 统计 Accuracy、F1 和 `on_time` Recall。评分前再次核对样本编号及标准答案，结果保存为 JSON 文件。

Test 只用于报告最终结果，不用于重新选择模型或调整参数。

In [ ]:
import json

# 读取最终 Test 预测，并核对全部 2,000 个编号
with test_prediction_path.open(encoding="utf-8") as file:
    test_predictions = [json.loads(line) for line in file]

test_lookup = {row["sample_id"]: row for row in test_rows}
ids = [row["sample_id"] for row in test_predictions]
assert len(ids) == len(set(ids)) == 2000
assert set(ids) == set(test_lookup)

fields = [
    "load_id", "truck_id", "origin_city", "destination_city",
    "revenue_usd", "fuel_used_gallons",
    "scheduled_delivery", "actual_delivery",
]

a_count = a_json = a_structure = a_exact = a_fields = 0
null_total = null_correct = 0
b_gold, b_pred = [], []

for row in test_predictions:
    source = test_lookup[row["sample_id"]]
    assert row["task"] == source["task"]
    gold = json.loads(source["messages"][-1]["content"])
    assert json.loads(row["reference"]) == gold

    try:
        predicted = json.loads(row["prediction"])
        valid_json = True
    except (json.JSONDecodeError, TypeError):
        predicted = None
        valid_json = False

    if row["task"] == "A":
        a_count += 1
        a_json += valid_json
        correct_shape = isinstance(predicted, dict) and set(predicted) == set(fields)
        a_structure += correct_shape
        a_exact += correct_shape and predicted == gold

        for field in fields:
            correct = (
                isinstance(predicted, dict)
                and field in predicted
                and predicted[field] == gold[field]
            )
            a_fields += correct
            if gold[field] is None:
                null_total += 1
                null_correct += correct
    else:
        b_gold.append(gold["delay_label"])
        label = (
            predicted["delay_label"]
            if isinstance(predicted, dict)
            and set(predicted) == {"delay_label"}
            and predicted["delay_label"] in {"delayed", "on_time"}
            else "__invalid__"
        )
        b_pred.append(label)

assert a_count == len(b_gold) == 1000

def label_scores(gold, predicted, label):
    # 计算单个类别的 Precision、Recall、F1
    tp = sum(g == label and p == label for g, p in zip(gold, predicted))
    fp = sum(g != label and p == label for g, p in zip(gold, predicted))
    fn = sum(g == label and p != label for g, p in zip(gold, predicted))
    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    return {"precision": precision, "recall": recall, "f1": f1}

delayed = label_scores(b_gold, b_pred, "delayed")
on_time = label_scores(b_gold, b_pred, "on_time")

test_metrics = {
    "model": "E1",
    "split": "test",
    "samples": 2000,
    "test_sha256": test_hash,
    "A_json_valid": f"{a_json}/1000",
    "A_structure": f"{a_structure}/1000",
    "A_field_accuracy": f"{a_fields}/8000",
    "A_null_accuracy": f"{null_correct}/{null_total}",
    "A_exact_records": f"{a_exact}/1000",
    "B_accuracy": f"{sum(g == p for g, p in zip(b_gold, b_pred))}/1000",
    "B_delayed_F1": delayed["f1"],
    "B_on_time_F1": on_time["f1"],
    "B_macro_F1": (delayed["f1"] + on_time["f1"]) / 2,
    "B_on_time_recall": on_time["recall"],
    "B_invalid_outputs": b_pred.count("__invalid__"),
}

test_metrics_path = eval_dir / "E1_final_test_metrics.json"
test_metrics_path.write_text(
    json.dumps(test_metrics, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps(test_metrics, ensure_ascii=False, indent=2))
print("指标文件：", test_metrics_path)

### 最终 Test 结果与结论

我们先在固定 Val 1,000 上比较 E1/E2/E3，并选定 **E1（Train 1,600，LoRA Rank 16）**。锁定模型和生成设置后，才对封存的合成 Test 2,000 进行一次生成与评分；Test 没有参与选模型。

| 指标 | E1 Val | E1 最终 Test |
| --- | ---: | ---: |
| Task A JSON 有效率 | 500/500 | **1000/1000** |
| Task A 字段准确率 | 4000/4000 | **8000/8000** |
| Task A 空值准确率 | 197/197 | **384/384** |
| Task A 整条记录完全正确 | 500/500 | **1000/1000** |
| Task B Accuracy | 497/500（99.4%） | **1000/1000（100%）** |
| Task B Macro-F1 | 0.9932 | **1.0000** |
| Task B `on_time` Recall | 0.9818 | **1.0000** |

Test 的 1,000 条 Task B 中，`delayed` 为 671 条、`on_time` 为 329 条；两类均全部判断正确。2,000 条预测编号完整、无重复，且标准答案与封存数据一致。

**结论与边界**：E1 在这份封存的合成 Test 上达到满分，说明它能很好地处理当前数据生成方式和题目范围。这个结果不等于真实物流文本或所有时间边界情况都能达到 100%。Phase 4 的人工困难样本须单独报告：Task A 为 400/400 字段正确，Task B 为 45/50，5 条错误属于同一类“晚 1 微秒”案例。压力测试不能并入正式 Test 成绩。

### 4.6 保存最终评测证据

归档 E1/E2/E3 的 Validation 预测与指标、E1 的最终 Test 预测与指标，以及模型选择记录和训练曲线。各模型权重已保存在各自结果包中，本包只保存评测证据。

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import hashlib
import json

evidence_files = [
    selection_path,
    metrics_path,
    test_metrics_path,
    loss_plot_path,
    prediction_dir / "E1_val_predictions.jsonl",
    prediction_dir / "E2_val_predictions.jsonl",
    prediction_dir / "E3_val_predictions.jsonl",
    test_prediction_path,
]

# 文件必须全部存在，避免生成不完整的结果包
for path in evidence_files:
    assert path.is_file(), f"缺少评测文件：{path}"

manifest = {
    "project": "FleetMind-LLM",
    "stage": "Phase 5 final evaluation",
    "selected_model": "E1",
    "selection_based_on": "Validation 1,000",
    "final_test": "2,000 synthetic instructions; evaluated once after selection",
    "files": {
        str(path.relative_to(eval_dir)): {
            "sha256": sha256_file(path),
            "bytes": path.stat().st_size,
        }
        for path in evidence_files
    },
}

zip_path = Path("/content/FleetMind_LLM_Phase5_Evaluation_Results.zip")
with ZipFile(zip_path, "w", ZIP_DEFLATED) as archive:
    for path in evidence_files:
        archive.write(
            path,
            f"FleetMind_LLM_Phase5_Evaluation/{path.relative_to(eval_dir)}",
        )
    archive.writestr(
        "FleetMind_LLM_Phase5_Evaluation/manifest.json",
        json.dumps(manifest, ensure_ascii=False, indent=2),
    )

# 回读检查压缩包是否完整
with ZipFile(zip_path) as archive:
    assert archive.testzip() is None

print("评测结果包：", zip_path)
print("包含文件数：", len(evidence_files) + 1)
print("大小：", round(zip_path.stat().st_size / 1024**2, 2), "MB")

In [ ]:
from google.colab import files

files.download(str(zip_path))

## FleetMind-LLM｜Phase 5 简明总结

**本阶段做了什么？** 用相同的 LoRA 参数训练三个数据规模：E1（1,600 条）、E2（4,000 条）、E3（8,000 条），在同一份 1,000 条 Validation 上比较，选定 E1 后，才对封存的 2,000 条 Test 做一次最终评测。

| 模型 | Task A：字段正确 | Task B：判断正确 | 结论 |
| --- | ---: | ---: | --- |
| E1 | 4,000/4,000 | 497/500 | 与 E2 并列，训练成本更低，选为最终模型 |
| E2 | 4,000/4,000 | 497/500 | 增加数据后，未见指标提升 |
| E3 | 4,000/4,000 | 437/500 | Task B 明显下降 |

**最终 Test 结果（E1）：** Task A 的 JSON 格式、结构及整条记录均为 **1,000/1,000** 正确，8 个字段合计 **8,000/8,000** 正确，空值 **384/384** 正确。Task B 为 **1,000/1,000** 正确；两个类别的 F1、Macro F1 和 on_time Recall 均为 **1.0**。这些是已完成的实测结果。

**错误案例说明：** E1 在 Validation 的 Task B 错了 3 条，都是把 *on_time* 判断成 *delayed*，但不能把这 3 条都归为“晚 1 微秒”。E3 错了 63 条，其中大量错误集中在一种改变时间叙述顺序的题目上，说明扩大训练集后，它对这种表述的判断反而变差。此前另做的 100 条人工困难样本中，Task B 为 **45/50**；5 条错误属于“晚 1 微秒”案例。

**如何理解泛化？** E1 在封存 Test 上表现很好，说明它能处理本项目同一数据来源和任务规则下的未见样本。但 Test 仍来自同一套数据构造流程；人工困难样本也暴露了极细时间边界的问题。因此，**不能把 Test 的满分解释为真实物流场景中也会零错误**。